# Correcciones entrega 2

Este notebook aplica las correcciones sobre el dataset final `Argenprop_limpio.csv`:

- agrega el cluster de barrio usando la logica de `5.EDA_Y_Clusters.ipynb`;
- renombra las variables segun su origen: `original_`, `imputada_`, `sintetica_` o `enriquecida_`;
- guarda el CSV limpio actualizado y un diccionario de variables.


In [1]:
from pathlib import Path
import re
import unicodedata

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

CLEAN_PATH = Path("Argenprop_limpio.csv")
DICT_PATH = Path("diccionario_variables_limpio.csv")

df = pd.read_csv(CLEAN_PATH, encoding="utf-8-sig")
print(df.shape)
df.head()

(7245, 55)


,sintetica_id_registro,imputada_Precio,imputada_Expensas,original_Calle,original_Altura,imputada_Piso,original_Link,imputada_Ambientes,imputada_Dormitorios,original_Banos,...,enriquecida_Colegios_500m,enriquecida_Dist_Comisaria_m,enriquecida_Dist_Gimnasio_m,enriquecida_Dist_Supermercado_m,enriquecida_Supermercados_500m,enriquecida_Dist_Avenida_m,enriquecida_Avenida_cercana,enriquecida_Paradas_colectivo_300m,imputada_Antiguedad_imputada,sintetica_Cluster
0,1,150000.0,260000.0,Bulnes,1600.0,No disponible,https://www.argenprop.com/departamento-en-vent...,3,2,1.0,...,10,493.126679,51.727490,20.279331,10,156.788297,Avenida Santa Fe,19,0,4
1,2,330000.0,203300.0,ARAOZ,1200.0,8,https://www.argenprop.com/departamento-en-vent...,4,3,2.0,...,11,403.114757,429.584392,249.248104,4,96.514964,Avenida Raúl Scalabrini Ortiz,10,1,4
2,3,270000.0,300000.0,Honduras,3900.0,2,https://www.argenprop.com/departamento-en-vent...,4,2,2.0,...,7,891.145341,383.017854,75.243729,7,425.495375,Avenida Coronel Niceto Vega,5,0,4
3,4,570000.0,1000000.0,Castex,3300.0,No disponible,https://www.argenprop.com/departamento-en-vent...,4,3,3.0,...,4,804.138281,286.697993,217.075594,3,108.444119,Avenida Casares,7,0,4
4,5,98000.0,150000.0,GURRUCHAGA,2100.0,5,https://www.argenprop.com/departamento-en-vent...,1,1,1.0,...,5,461.483469,393.568008,85.286744,6,378.802881,Avenida Raúl Scalabrini Ortiz,7,0,4


## Cluster de barrio

Se replica el criterio de `5.EDA_Y_Clusters.ipynb`: se agrupa por barrio, se calculan promedios de variables inmobiliarias, se estandarizan y se aplica KMeans con 6 clusters.


In [2]:
prefixes = ("original_", "imputada_", "sintetica_", "enriquecida_")

if any(str(col).startswith(prefixes) for col in df.columns):
    print("El archivo ya tiene prefijos. No se vuelve a calcular para evitar doble prefijo.")
else:
    features_cluster = [
        "Precio", "Expensas", "Sup_Cubierta_m2", "Sup_Total_m2",
        "Ambientes", "Dormitorios", "Baños", "Antiguedad"
    ]
    features_cluster = [col for col in features_cluster if col in df.columns]

    barrio_features = (
        df.groupby("Barrio")[features_cluster]
        .mean(numeric_only=True)
        .dropna()
    )

    scaler = StandardScaler()
    X = scaler.fit_transform(barrio_features)

    kmeans = KMeans(n_clusters=6, random_state=42, n_init=10)
    barrio_features = barrio_features.copy()
    barrio_features["Cluster"] = kmeans.fit_predict(X)

    df["Cluster"] = df["Barrio"].map(barrio_features["Cluster"]).astype("Int64")
    print("Cluster agregado. NAs:", df["Cluster"].isna().sum())

    barrio_features.sort_values("Cluster")

El archivo ya tiene prefijos. No se vuelve a calcular para evitar doble prefijo.


## Renombre por origen de variable

Criterio usado:

- `original_`: variables observadas directamente y no imputadas en el pipeline final.
- `imputada_`: variables que fueron completadas o imputadas durante limpieza.
- `sintetica_`: indicadores derivados por reglas, ids de exportacion y cluster.
- `enriquecida_`: variables agregadas por geocodificacion o cruces geoespaciales.


In [3]:
COLUMNAS_IMPUTADAS = {
    "Precio", "Expensas", "Piso", "Ambientes", "Dormitorios", "Antiguedad",
    "Estado", "Disposicion", "Tipo_Unidad", "Sup_Cubierta_m2", "Sup_Total_m2",
    "Antiguedad_imputada"
}

COLUMNAS_SINTETICAS = {
    "Unnamed_0", "id_registro", "Aire_acondicionado_individual", "Losa_radiante",
    "Gas_natural", "Agua_corriente", "Balcon", "Terraza", "Jardin", "Patio",
    "Baulera", "Cochera", "Muebles_de_cocina", "Permite_Mascotas", "Ascensor",
    "Pileta", "Parrilla", "Gimnasio", "Sauna", "Laundry", "Vigilancia", "Cluster"
}

COLUMNAS_ENRIQUECIDAS = {
    "Latitud", "Longitud", "Barrio", "Comuna", "Dist_Subte_m", "Subte_cercano",
    "Linea_subte", "Dist_Hospital_m", "Hospital_cercano", "Dist_Colegio_m",
    "Colegios_500m", "Dist_Comisaria_m", "Dist_Gimnasio_m", "Dist_Supermercado_m",
    "Supermercados_500m", "Dist_Avenida_m", "Avenida_cercana", "Paradas_colectivo_300m"
}

RENOMBRES_BASE = {"Unnamed: 0": "id_registro"}

def normalizar_nombre(columna):
    nombre = RENOMBRES_BASE.get(str(columna), str(columna))
    nombre = unicodedata.normalize("NFKD", nombre)
    nombre = nombre.encode("ascii", "ignore").decode("ascii")
    nombre = re.sub(r"[^0-9A-Za-z_]+", "_", nombre)
    nombre = re.sub(r"_+", "_", nombre).strip("_")
    return nombre

def tipo_variable(columna):
    nombre = normalizar_nombre(columna)
    if nombre in COLUMNAS_IMPUTADAS or nombre.endswith("_imputada"):
        return "imputada"
    if nombre in COLUMNAS_ENRIQUECIDAS:
        return "enriquecida"
    if nombre in COLUMNAS_SINTETICAS:
        return "sintetica"
    return "original"

if any(str(col).startswith(prefixes) for col in df.columns):
    df_renombrado = df.copy()
    tipos = [str(col).split("_", 1)[0] for col in df.columns]
    nombres_sin_prefijo = [str(col).split("_", 1)[1] for col in df.columns]
    nombres_sin_prefijo = ["Unnamed: 0" if nombre == "id_registro" else nombre for nombre in nombres_sin_prefijo]
    diccionario = pd.DataFrame({
        "variable_original": nombres_sin_prefijo,
        "tipo_variable": tipos,
        "variable_renombrada": df.columns
    })
else:
    renombres = {col: f"{tipo_variable(col)}_{normalizar_nombre(col)}" for col in df.columns}
    if len(set(renombres.values())) != len(renombres):
        raise ValueError("Hay nombres duplicados luego de renombrar")

    diccionario = pd.DataFrame({
        "variable_original": list(renombres.keys()),
        "tipo_variable": [tipo_variable(col) for col in renombres.keys()],
        "variable_renombrada": list(renombres.values())
    })
    df_renombrado = df.rename(columns=renombres)

diccionario["tipo_variable"].value_counts().sort_index()

tipo_variable
enriquecida    18
imputada       12
original        4
sintetica      21
Name: count, dtype: int64

In [4]:
df_renombrado.to_csv(CLEAN_PATH, index=False, encoding="utf-8-sig")
diccionario.to_csv(DICT_PATH, index=False, encoding="utf-8-sig")

print(f"Archivo actualizado: {CLEAN_PATH}")
print(f"Diccionario generado: {DICT_PATH}")
print("Shape:", df_renombrado.shape)
print("Cluster NA:", df_renombrado["sintetica_Cluster"].isna().sum() if "sintetica_Cluster" in df_renombrado.columns else "sin columna")
df_renombrado.head()

Archivo actualizado: Argenprop_limpio.csv
Diccionario generado: diccionario_variables_limpio.csv
Shape: (7245, 55)
Cluster NA: 0


,sintetica_id_registro,imputada_Precio,imputada_Expensas,original_Calle,original_Altura,imputada_Piso,original_Link,imputada_Ambientes,imputada_Dormitorios,original_Banos,...,enriquecida_Colegios_500m,enriquecida_Dist_Comisaria_m,enriquecida_Dist_Gimnasio_m,enriquecida_Dist_Supermercado_m,enriquecida_Supermercados_500m,enriquecida_Dist_Avenida_m,enriquecida_Avenida_cercana,enriquecida_Paradas_colectivo_300m,imputada_Antiguedad_imputada,sintetica_Cluster
0,1,150000.0,260000.0,Bulnes,1600.0,No disponible,https://www.argenprop.com/departamento-en-vent...,3,2,1.0,...,10,493.126679,51.727490,20.279331,10,156.788297,Avenida Santa Fe,19,0,4
1,2,330000.0,203300.0,ARAOZ,1200.0,8,https://www.argenprop.com/departamento-en-vent...,4,3,2.0,...,11,403.114757,429.584392,249.248104,4,96.514964,Avenida Raúl Scalabrini Ortiz,10,1,4
2,3,270000.0,300000.0,Honduras,3900.0,2,https://www.argenprop.com/departamento-en-vent...,4,2,2.0,...,7,891.145341,383.017854,75.243729,7,425.495375,Avenida Coronel Niceto Vega,5,0,4
3,4,570000.0,1000000.0,Castex,3300.0,No disponible,https://www.argenprop.com/departamento-en-vent...,4,3,3.0,...,4,804.138281,286.697993,217.075594,3,108.444119,Avenida Casares,7,0,4
4,5,98000.0,150000.0,GURRUCHAGA,2100.0,5,https://www.argenprop.com/departamento-en-vent...,1,1,1.0,...,5,461.483469,393.568008,85.286744,6,378.802881,Avenida Raúl Scalabrini Ortiz,7,0,4
